# Assignment 04: Generalization and Regularization Techniques for Neural Networks
---

**Due Date:** Tuesday 06/16/2026 (by midnight)

**Please fill these in before submitting, just in case I accidentally mix up file names while grading**:

Name: Jane Hacker

CWID-5: (Last 5 digits of cwid)

# Introduction 

Welcome to our next assignment on regularization techniques to improve
generalization performance in neural networks.  Deep Learning models have so much flexibility and capacity that **overfitting can be a serious problem**, if the training dataset is not big enough. Sure it does well on the training set, but the learned network **doesn't generalize to new examples** that it has never seen!

We will be continuing from the last assignment and looking at how we can improve the generalization performance of a neural network model by using
regularization.  Also we will have a bit of practice overriding the Keras `Callback` class and using Keras callbacks in this assignment.


**Instructions:**

- Do not use loops (for/while) in your code, unless the instructions explicitly ask you to do so.
- Use the Keras functional API to implement your asked for network architectures, and to practice and
  get used to specifying models using that API.
- The workflow has changed a bit in this assignment.  You have not been given the function
  definitions for the functions you will write in the `assg_tasks.py` file.  Make sure you correctly
  declare the function, and always provide PyDoc function documentation for your function.
  See [Python Docstrings](https://www.geeksforgeeks.org/python-docstrings/) for details on using them
  and best practices.  You are required to use the **NumpyDoc style Docstrings**, where you document the
  input Parameters and the Return values to each function.
- Also because of this change, you will need to uncomment calls to import your function once you create it and
  uncomment the code to call the tests on your function.  Your functions need to be named exactly as required
  for the task so that the tests given for functions do not need to be modified to test your code.
- Also see [Pep8 Python Style Guide](https://peps.python.org/pep-0008/) and follow
  best practices for formatting your Python code.

**You will learn:**

- to use regularization in your deep learning models.
- the effect of different regularization techniques on performance.
- what weight and Dropout regularization are and how you use them in your Keras models
- How to use and create Keras callbacks by subclassing the `Callback` object from Keras

# Packages

The following imports should be all of the packages that you will need for this assignment.
We are using the Keras API in this assignment and in future assignments, so the `tensorflow` and `keras` modules
you need are now available in the notebook.

In [ ]:
# assignment wide imports go here, usually all of your imports for notebooks should
# be put up at the top here, if they were not given to you at the start of the assignment
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from mlxtend.plotting import plot_decision_regions

In [ ]:
# The following ipython magic will reload changed file/modules.
# So when editing function in source code modules, you should
# be able to just rerun the cell, not restart the whole kernel.
%load_ext autoreload
%autoreload 2

The imports of the function you will write have been commented out here this time.  You will need to uncomment
the imports once you declare and write your functions here, and also in the `src/test_assg_tasks.py` file to
run the unit tests on your work.

In [ ]:
# assignment function imports for doctests and github autograding
# these are required for assignment autograding
# uncomment these imports as you declare and implement the assg_task functions
# NOTE: The following imports require that the PYTHONPATH be correctly configured in your jupyter lab environment
from assg_utils import run_unittests, run_doctests
#from assg_tasks import EveryNEpochs
#from assg_tasks import get_unregularized_model
#from assg_tasks import get_l2_regularized_model
#from assg_tasks import get_dropout_model
from assg_tasks import load_soccer_dataset
from assg_tasks import plot_history

# Overview of the Dataset

**Problem Statement**: You hae just been hired
as an AI expert by the French Soccer League.  They would like you to recommend positions
where France's goal keeper should kick the ball so that the French team's players can then
hit it with their head.

<img src="../figures/field.png" style="width:900px;height:600px;">
<caption><center> <u> Figure 1 </u>: Football field<br> The goal keeper kicks the ball in the air, the players of each team are fighting to hit the ball with their head </center></caption>

In [ ]:
# Loading the flower dataset
train_X, train_y, test_X, test_y = load_soccer_dataset()

In [ ]:
# positive classes are when French player got the ball
french_X = train_X[train_y == 1]
opposition_X = train_X[train_y == 0]
plt.figure(figsize=(12,8))
plt.scatter(french_X[:,0], french_X[:,1], color="blue", label='French header')
plt.scatter(opposition_X[:,0], opposition_X[:,1], color="red", label='opposition header')
plt.xlim(-0.65, 0.65)
plt.xlabel('soccer field length position')
plt.ylabel('width position')
plt.grid()
plt.legend();

Each dot corresponds to a position on the football field where a football player has hit
the ball with his/her head after the French goal keeper has shot the ball from the left
side of the football field.
- If the dot is blue, it means the French player managed to hit the ball with his/her head
- If the dot is red, it means the other team's player hit the ball with their head

**Your goal**: Use a deep learning model to find the positions on the field where the goalkeeper should kick the ball.

**Analysis of the dataset**: This dataset is a little noisy, but it looks like a diagonal
line separating the upper left half (blue) from the lower right half (red) would work well. 

You will first try a non-regularized model. Then you'll learn how to regularize it and
decide which model you will choose to solve the French Soccer League's problem. 

But first, lets practice creating a `Callback` that you can use when evaluating your models
performance in this assignment.

# Task 1: EveryNEpochs Callback

You will again need to train your models for more than a couple hundred epochs of trainings in many cases in this assignment, in order
to see the effects of different network regularization techniques.  This may take a few minutes to complete each training loop.

The `fit()` function provides a `verbose` parameter that can take on values of 0, 1 and 2.  `verbose=1` or `verbose=2` will display the
current loss after each epoch of training using the fit progress bar callback.  `verbose=0` will turn off all reporting
of progress.  When we have more than a few 10s of epochs, the output becomes long and clutters up our notebooks.

**Task**: Implement a `keras.callbacks.Callbacks` subclass called `EveryNEpochs` in the `src/assg_tasks.py`.  This callback should
override the `on_epoch_end()` member method. You can assume that the `accuracy` metric has been compiled in so that you will have it
as a measurement to report.  The `logs` parameter passed into `on_epoch_end()` has a `get()` method to lookup keys in this dictionary
like object.  So you should assume you have the keys `loss`, `accuracy` for the training loss and accuracy, and `val_loss`, `val_accuracy`
for the validation loss and accuracy.

Use a print statement to report the training loss, training accuracy, validation loss and validation accuracy on standard output.  Format
your output to look like the following:

```
Epoch 0010/1000: training loss - 0.1234  validation loss - 0.1234  training accuracy - 0.1234  validation accuracy - 0.1234
```

In addition, override the `__init__` constructor method for your subclass.  Accept an additional parameter `N`, which will be the
step size for reporting.  For example if N is 10, then you should only report results for epochs 0, 10, 20, 30, ...  Use
the modulus operator in your `on_epoch_end()` function something like this

```python
if epoch % self.N == 0:
    # then report loss and validation for this epoch
```

Also make sure you always report the last epoch's results, no matter how many epochs are are asked to be trained with, so you can
see the final values of the metrics.  You could use an if statement and repeat your print, or you could do something like:

```python 
# 0 based indexing, so if ask for 100 epochs, epochs are numbered 0-99
last_epoch = self.params['epochs'] - 1
if epoch % self.N or epoch == last_epoch:
    # then report metrics, and the or will guarantee always report the last epochs metrics as well
```

In [ ]:
### TESTED Subclass EveryNEpochs() callback
# uncomment when ready to run the unit tests for your callback
#run_unittests(['test_EveryNEpochs_callback'])

**Expected Result**: Use the following cell to test your output.  For N=16 you should get the following as the exact output

```
Epoch: 0000/0099  training loss - 3.1431  validation loss - 2.6056  training_accuracy - 0.1225  validation accuracy - 0.1000
Epoch: 0016/0099  training loss - 1.6049  validation loss - 2.6464  training_accuracy - 0.4812  validation accuracy - 0.1050
Epoch: 0032/0099  training loss - 0.8179  validation loss - 2.6284  training_accuracy - 0.8313  validation accuracy - 0.0950
Epoch: 0048/0099  training loss - 0.2455  validation loss - 4.3797  training_accuracy - 0.9600  validation accuracy - 0.1150
Epoch: 0064/0099  training loss - 0.1337  validation loss - 2.8541  training_accuracy - 0.9725  validation accuracy - 0.1200
Epoch: 0080/0099  training loss - 0.0795  validation loss - 3.1307  training_accuracy - 0.9762  validation accuracy - 0.1100
Epoch: 0096/0099  training loss - 0.0037  validation loss - 3.1678  training_accuracy - 1.0000  validation accuracy - 0.1250
Epoch: 0099/0099  training loss - 0.0033  validation loss - 3.2393  training_accuracy - 1.0000  validation accuracy - 0.1400
```

There are 2 spaces between each reported metric, and only 1 elsewhere.  Each metric is reported to 4 digits, and the epochs
are reported using 4 digits with 0's to fill in.  This is mostly so that each line ends up lining up with one another, to make
reading and following progression easier.

In [ ]:
# simple model and random data to test my callback
model = keras.Sequential([
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])
x = np.random.random((1000, 64*64))
y = np.random.randint(0, 10, size=(1000,))
y = to_categorical(y)

# Fit the model with the custom callback
history = model.fit(x, y, 
                    epochs=100, verbose=0, 
                    validation_split = 0.2,
                    #callbacks=[EveryNEpochs(16)] # uncomment this when ready to start testing your callback by hand
                   )

# Task 2: Unregularized model

We will first create a neural networks using the Keras functional API as in the previous
assignment.  We will start with a model with no regularization, then try adding regularization
to improve generalization performance.

**Task** Write a function that creates a `keras.Model` called `get_unregularized_model()`.
There are only 2 input features, the soccer field length and width position of the
data point where a header occurred.  This is again a binary classification problem
like we have seen.  So use the recommended loss and activation functions and final
sigmoid activation for the output layer.

Create your model to have three intermediate layers.  Put 1024 units in
the first hidden layer `hidden1`, followed by 512 then 128 units in `hidden2` and `hidden3`.
Remember to use `relu` activation for the intermediate layers.  Name
your model "unregularized_model", and your layers "input", "hidden1", "hidden2", "hidden3" and "output"
respectively.  The model should be compiled before being returned.  Use the standard loss function,
and make sure that you ask that `accuracy` metrics are evaluated for the model.


In [ ]:
### TESTED FUNCTION get_logistic_regression_model()
# uncomment these lines to run unittests on your function and get an instance called unreg_model
#run_unittests(['test_get_unregularized_model'])

#unreg_model = get_unregularized_model()

**Task**: In the following cell, show the summary of your unregularized model.
This is just another check for you that you are getting the model you expect
from your function to create it, and to get a feel for the number of trainable
parameters in this neural network.

**Expected Result**: You should find that your unregularized model has 4 layers and
593,665 trainable parameters in all.

In [ ]:
# display your unregularized model summary here

**Task**: Next train your model using the `fit()` function on the `train_X` inputs
and `train_y` labels.  We only have 211 samples in this data, but use the parameter to
the `fit()` function to ask it to randomly chose 10% of the training data for
validation.  Use a batch size of 64 for this data. Train on your logistic
regression model for 1000 epochs. Keep the returned history of the model fit
so that we can plot the learning curve results next.

Also test your `EveryNEpoch` callback some more.  Set `verbose=0` for the training loop, and instead
pass in your `EveryNEpoch` callback to report progress.  I suggest having reports every 25, 50 or
100 epochs of the 1000 epoch run as a good granularity to follow progress but not clutter your notebook
output with long cells.

Optional: Also if your training runs are taking more than a couple of minutes (or even if not),
you might find it useful to practice using the `TensorBoard` callback and monitoring your
training runs using a `TensorBoard.

**Expected Result**: You should see, using your `EveryNEpochs` callback, that validation loss 
steadily decreases close to 0, but validation loss will start increasing at some point.  Likewise
accuracy will probably reach 100% at some point on the training data, but you will see it stall and maybe
decline for validation data.

In [ ]:
# fit your unregularized model as described here

**Task**: The `plot_history()` function that we have used in our class lecture notebooks has been copied
to your `src/assg_tasks.py` file and imported.  Use the function to plot the training/validation
loss and the training/validation accuracy as demonstrated in our class lectures.

**Expected Result:** The same as mentioned when training, you should see evidence that this
model is overfitting.  The amount of data is small, so even using a batch size of 64, it can
be a bit noisy.  But you should clearly see that training and validation loss start to diverge,
that validation loss goes down for a bit but then begins getting worse at some point 
(typically sometime around or after epoch 200).  

Training accuracy should creep up and start hitting 100% at some point.  It may be tough to
see if validation accuracy remains steady (usually around 90%) or shows in downward trend.

In [ ]:
# use plot_history() here to visualize the learning curves

**Task**: Visualize the decision boundary made by your unregularized classification
model.  There is a method called `plot_decision_regions()` from the `mlxtend.plotting` library
that has been imported.  You can see [mlxtend Examples and Documentation here](https://rasbt.github.io/mlxtend/)
Use this function to visualize the decision boundary that is learned by your simple logistic
regression classifier.

**Expected Results**: You should see evidence of overfitting from the decision boundary
visualization.  You might need to or at least find it helpful to limit the x and y ranges
of the plot (`fig.set_xlim()` for example) to better see the data.  You should definitely
get a nonlinear decision boundary, and see a couple of places where the overfit model
carves out special cases to get its accuracy close to 100%.

In [ ]:
# use plot_decision_regions() here to visualize the resulting decision boundary of your model

**Task**: Now lets retrain the model using all the data (don't use any validation split), and evaluate its performance on
the training data, and on the held back test data.  What are you expecting you will see for accuracy on the training
and test data here?  This unregularized model can be thought of as a baseline model from this point on.  The model
clearly fits, and is able to generalize.  But we want to apply techniques now to improve generalization so that we can
get the best performance possible for typical datasets that we have not trained with and may see from the real world.

So do the following:

- You need to first make sure you get a new fresh copy of your unregularized model, by calling your `get_unregularized_model()` again,
  don't try and retrain from your already trained model.
- Retrain the new model using all of the data.  You can't use your callback since it expects validation metrics (but feel
  free if you are inclined to add in code to only report train metrics if validation is not available, as long as it
  continues to pass the given unit tests).
- Evaluate the performance of the model on both the training data and on the held back test data.  Fill out the table at the end
  of this task reporting on the training and validation accuracy of your unregularized model.
- Save the final version of your model to a file named `../models/unregularized_final_model.keras`, using the `model.save()` method,
  we will make use of this saved final model later.

In [ ]:
# get a fresh unregularized model to start training from scratch

# retrain on all data and evaluate on train and test


In [ ]:
# report loss and accuracy on all the data trained with for final evaluation

In [ ]:
# report loss and accuracy on the test data set for final evaluation

In [ ]:
# save the final model to file for later evaluation

## Final Report: Unregularized Model Performance

Fill in this table to report the final result you saw for your unregularized
model.

| Metric         | Result |
|----------------|--------|
| train loss     | 0.0000 |
| test  loss     | 0.0000 |
| train accuracy | 0.0000 |
| test  accuracy | 0.0000 |

# Task 3: Apply Early Stopping

Lets first see if we stop once we detect that the validation loss starts rising, if we see
much if any improvement on a final evaluation of the training data.  You will get some
more practice with callbacks in this task.

**Task**:  For this task you will be reusing your `get_unregularized_model()` function, so there are no
new functions to write and test.  Start by getting a fresh unregularized model again using your
function.

The performance is quite noisy here because of the small amount of data.  So where overfitting
occurs can vary a lot and is hard to predict.  So while we would like to compare apples to apples
and have all of our models in this assignment be trained using the full training set, for this task
we will make an exception.  

Probably the best model we can get with an early stopping approach would be to train a model
and use validation metrics, and keep the model that obtains the best result on validation loss
(we could also try this by keep best model on validation accuracy).

So for this task, train your freshly created model using all of the training data again
(no validation split), but otherwise using the same fit parameters, including training
for 1000 epochs.  But add in a `MonitorCheckpoint` callback. Use the `MonitorCheckpoint` to save the
model seen with the best validation loss.  Name this model `../models/early_stopped_model.keras`.

This best model is at a slight advantage since it is only trained with part of the training data, but
it will usually better capture the best place to perform early stopping.

**Expected Result**:  The best validation loss is usually around 0.2 or a bit under.  Which epoch this occurs can
vary a lot depending on the random splits, but usually should happen sometime after epoch 100.

If you are interested in knowing the epoch and validation loss, you can use verbose=1 or your own
callback to display the metrics report for every epoch of training, though that is a bit tedious.

You can also specify a `filepath` for your `ModelCheckpoint` something like this:

```python
filepath='../models/early_stopped_model_epoch-{epoch:04d}-val_loss-{val_loss:04f}.keras'
```

This will produce a different name each time a point is reached where the validation loss is the best seen
so far.  But the last one will be the one with the lowest loss, and you can see the epoch where that happened
from the file name.

Also alternatively, you can use the TensorBoard API, which has interactive graphs, to find the point on the 
validation loss curve where the minimum is reached.


In [ ]:
# get a fresh unregularized model to start training from scratch

# setup up callbacks, ModelCheckpoint to save model with best val_loss and EarlyStopping to stop when val_loss stops increasing

# retrain on all data and evaluate on train and test


**Task**: Now lets evaluate the best checkpointed model that you achieved using all of the training and test data, like you did
in the previous task for your unregularized model.

First load back in the best saved model checkpoint using `keras.models.load_model()' here.

**Note**:  Make sure that you have a filename in `../models` that does not change each time the notebook is rerun, or else when
run in the autograder, different file name checkpoints will be produced and the next cell might not run.

**Expected Result**: You should see that when the early stopped model is evaluated on all of the test data, its performance will decline
(typically accuracy is 94% or so).  Again this is a bit unfair for this model since it overfit on only part of the training data,
so the evaluation on all of the training data is seeing some unseen samples now.

But usually there should be a small, but probably significant, increase in accuracy for the early stopped model
on the test accuracy.

In [ ]:
# load back in the best early stopped model

And as before, report this early stopped models performance on all of the training data, and on the test data.  Then
fill out the table noting the performance you see on the test data for the early stopped model.

In [ ]:
# report loss and accuracy on all the training data for this model

In [ ]:
# report loss and accuracy on the test data set for final evaluation

## Final Report: Early Stopped Model Performance

Fill in this table to report the final result you saw for your unregularized
model.

| Metric         | Result |
|----------------|--------|
| train loss     | 0.0000 |
| test  loss     | 0.0000 |
| train accuracy | 0.0000 |
| test  accuracy | 0.0000 |

# Task 4: L2 Regularization

You will not try and use L2 weight regularization to help the generalization performance of your network.  Recall that L2 regularization acts
as a penalty term on the loss function being optimized.  L2 regularization uses the sum of the square of all of the trainable weights
of the network for the penalty.  So if a weight is small or 0, then the square of the weight will be small.  But if a weight is large, the square
of the weight will be even larger.  Adding this penalty encourages the optimizer to find solutions that work but work with as small of weights as possible.
Formally, the L2 regularization penalty can be defined as follows:

\begin{equation}
J_{regularized} = \small \underbrace{J_{loss}}_\text{loss function} + \underbrace{\frac{1}{m} \frac{\lambda}{2} \sum\limits_l\sum\limits_k\sum\limits_j W_{k,j}^{[l]2} }_\text{L2 regularization penalty}
\end{equation}

Here $J_{loss}$ is whatever loss function being optimized, for example `binary_crossentropy`.  The L2 regularization penalty is summed over all $l$ layers of the
network.  Each layer has a 2-D weight matrix of size $k$ by $j$ typically.  The square of all weights in all layers is thus summed up and added to the loss
function.  

The Lambda $\lambda$ parameter controls the amount of L2 regularization applied.  When Lambda is 0, then no penalty is applied, i.e. the loss function is unregularized.
The large the value of Lambda, the bigger the penalty.  So if Lambda is too small, there is not enough penalty and the model may keep overfitting.  But if Lambda
is too large, too much emphasis is given on making the weights small or 0, and the model will begin underfitting.

**Task**:  Implement a function named `get_l2_regularization_model()`.  This function should use the same architecture as before (3 hidden layers with
1024, 512 and 128 units respectively). But add in L2 regularization to each of the intermediate layers  Add a parameter named
`lmbda` to be passed in to your function.  This parameter should be the amount of L2 regularization that you apply to all of your
3 intermediate layers.  The function should create and compile the model as before and return it.

**Note**: `lambda` is a reserved keyword in Python, so it is suggested you name the parameter `lmbda` for this functions input.


In [ ]:
### TESTED FUNCTION get_l2_regularized_model()
# uncomment these lines to run unittests on your function and get an instance called l2_reg_model
#run_unittests(['test_get_l2_regularized_model'])

#l2_reg_model = get_l2_regularized_model(0.000)

**Task**: In the following cell, show the summary of your L2 regularized model.

**Expected Result**: You should find that your unregularized model has 4 layers and
593,665 trainable parameters in all, the same as your first unregularized model.
You cannot tell, but there should be L2 regularization applied now on all of the
intermediate layers.

In [ ]:
# display your l2 regularized model summary here

**Task**: Next train your model using the `fit()` function on the `train_X` inputs
and `train_y` labels.  You should use the same values you used previously to fit the unregularized
and early stopped models. Epochs is 1000, batch_size 64, and use a 10% validation split.

**Expected Result**: You should see, using your `EveryNEpochs` callback, that training loss, if you
are using enough L2 regularization, does not go to 0, and will level off at some point.  Also validation
loss with enough L2 regularization will not start rising back up again after some point.

If the `lambda` parameter is too small, then the model will still overfit, and training loss will continue towards
0 and validation loss will begin rising again.

If the `lambda` parameter is too big, then the model will be underfitting.  This may be hard to detect, but if training
loss levels off at a value larger than you have seen, and if training accuracy is lower than the validation accuracy
you saw before, then the model is underfitting.

In [ ]:
# fit your l2 regularized model as described here

**Task**: Plot your learning curves for your L2 regularized training run again as usual using the
`plot_history()` 

**Expected Result:** The results will depend on the value of `lambda` you are using, but you should be able to begin
to recognize signs of underfitting or overfitting.  Remember that larger values of `lambda` make the penalty larger and
drive the model to underfit.  But too small values of `lambda` may let the model start overfitting again.

In [ ]:
# use plot_history() here to visualize the learning curves

**Task**: Visualize the decision boundary made by your unregularized classification
model as usual.

**Expected Results**: You don't usually have such a tool to help determine if overfitting or underfitting may be occurring.
But here since we can visualize the decision boundary.  If there is too much wiggling, and still some carved out spots
in the decision space to memorize specific points in the training data, then overfitting may be happening.  A good `lambda` will
usually remove most all of the special carved out spots in the visualization of the decision boundary here.

In [ ]:
# use plot_decision_regions() here to visualize the resulting decision boundary of your model

Spend a little bit of time trying some different values of `lambda` on the previous few cells.  Make sure you don't confuse yourself, you need
to ensure you always get a fresh model with some value of `lambda` before you try and retrain.

In my experience, values of `lambda` around `0.01` or bigger are too big, the model will be underfitting.  But values at `0.0001`
or smaller are too small, overfitting will still be happening a lot.

Try those values and a few around `0.001`.  Pick a value that you think is "just right" doesn't seem to be overfitting, but is not 
underfitting either.  Once you have a value in mind, retrain again a fresh L2 regularized model on all of the available training data.

**Task**: Retrain the your L2 regularized model using your selected `lambda` using all the data (don't use any validation split),
and evaluate its performance on the training data, and on the held back test data.  If you pick a good value, usually you will see a bit
of an improvement over your early stopped model when looking at validation accuracy (though the dataset is small so the signal can be
noisy, so don't get too hung up, redoing the following steps will get slightly different final evaluations).

So do the following:

- You need to first make sure you get a new fresh copy of your l2 regularized model, by calling your `get_l2_regularized_model()`, makes sure
  you specify your selected L2 `lambda` amount you want to evaluate with.
- Retrain the new model using all of the data.  You can't use your callback since it expects validation metrics (but feel
  free if you are inclined to add in code to only report train metrics if validation is not available, as long as it
  continues to pass the given unit tests).
- Evaluate the performance of the L2 regularized model on both the training data and on the held back test data.  Fill out the table at the end
  of this task reporting on the training and validation accuracy of your L2 regularized model.
- Save the final version of your model to a file named `../models/l2-regularized-model.keras`, using the `model.save()` method,
  we will make use of this saved final model later.

In [ ]:
# get a fresh l2 regularized model to start training from scratch

# retrain on all data and evaluate on train and test


In [ ]:
# report loss and accuracy on all the data trained with for l2 regularized final evaluation

In [ ]:
# report loss and accuracy on test data for l2 regularized final evaluation

In [ ]:
# save the final model to file for later evaluation

## Final Report: L2 Regularized Model Performance

Fill in this table to report the final result you saw for your L2 regularized
model.

| Metric         | Result |
|----------------|--------|
| L2 lambda      | 0.0000 |
| train loss     | 0.0000 |
| test  loss     | 0.0000 |
| train accuracy | 0.0000 |
| test  accuracy | 0.0000 |

# Task 5: Dropout Regularization 

Finally for this assignment lets apply dropout regularization to our unregularized network.

**Task**:  Implement a function named `get_dropout_model()`.  This function should use the same architecture as before (3 hidden layers with
1024, 512 and 128 units respectively).  But add in a `Dropout` layer after each of the intermediate hidden layers.  Add a parameter named
something like `dropout_rate` for this function.  Use it as the dropout rate for all 3 of your `Dropout` layers in your model.

**Note**: Be careful when using the Functional API in this function that the previous hidden layer is input to the dropout layer, and the dropout layer
is now input to the next layer in the network.  E.g. it is easy to copy/paste error here and add in dropout layers but forget to update layer inputs
correctly.

In [ ]:
### TESTED FUNCTION get_dropout_model()
# uncomment these lines to run unittests on your function and get an instance called drop_model
#run_unittests(['test_get_dropout_model'])

#drop_model = get_dropout_model(0.0)

**Task**: In the following cell, show the summary of your L2 regularized model.

**Expected Result**: You should find that your unregularized model has 7 layers now
because of the 3 additional dropout layers.  Surprisingly the number of trainable
parameters should be the same as before `593,665`.  Do you know why the
number of trainable parameters has not increased here?

In [ ]:
# display your dropout regularized model summary here

**Task**: Next train your model using the `fit()` function on the `train_X` inputs
and `train_y` labels.  You should use the same values you used previously to fit the previous models.
Epochs is 1000, batch_size 64, and use a 10% validation split.

**Expected Result**: You should see, using your `EveryNEpochs` callback, that training loss will stop
decreasing at some point with a good dropout rate, usually below 0.2.

You may not have an intuitive feel for a good dropout rate here.  In my testing for this dataset, 0.5 is way to low, the
model will be still overfitting.  Try dropout rates around 0.75 or so.

In [ ]:
# fit your dropout model as described here

**Task**: Plot your learning curves for your L2 regularized training run again as usual using the
`plot_history()` 

**Expected Result:** The results will depend on the dropout rate you are using.  Good values for the rate should
give similar results as before.  You want a rate that looks like it is balancing between overfitting and
underfitting, maybe leaning just a bit towards still overfitting a bit, with training loss maybe decreasing
a bit after its initial large drop over the 1000 training epochs.

In [ ]:
# use plot_history() here to visualize the learning curves

**Task**: Visualize the decision boundary made by your unregularized classification
model as usual.

**Expected Results**:  Visualizing the decision boundary can help you understand the
effects your dropout regularization is having.  As with L2 regularization, usually
a good value of dropout will eliminate all obvious carve outs to memorize
small parts of the space, though the decision boundary line will curve a bit in places that
make sense given the training data being used.

In [ ]:
# use plot_decision_regions() here to visualize the resulting decision boundary of your model

Spend a little bit of time seeing the effects dropout rates have.  As mentioned before, around 0.75 is probably about right.  But try 0.5 and 0.95
to see effects of still overfitting and underfitting again respectively.

**Task**: Retrain the your dropout regularized model using your selected dropout rate using all the data (don't use any validation split),
and evaluate its performance on the training data, and on the held back test data.  If you pick a good value, you should match L2 regularization,
or maybe improve a bit.  Again the dataset is small so the signal is noisy here, so don't get too hung up.  If you have time, you can try rerunning
with the same dropout rate and L2 regularization lambda to see how much the values can vary between runs.

So do the following:

- You need to first make sure you get a new fresh copy of your dropout regularized model, by calling your `get_dropout_model()`, makes sure
  you specify your selected dropout rate amount you want to evaluate with.
- Retrain the new model using all of the data.  You can't use your callback since it expects validation metrics (but feel
  free if you are inclined to add in code to only report train metrics if validation is not available, as long as it
  continues to pass the given unit tests).
- Evaluate the performance of the dropout regularized model on both the training data and on the held back test data.  Fill out the table at the end
  of this task reporting on the training and testing accuracy of your dropout regularized model.
- Save the final version of your model to a file named `../models/dropout_model.keras`, using the `model.save()` method,
  we will make use of this saved final model later.

In [ ]:
# get a fresh dropout regularized model to start training from scratch

# retrain on all data and evaluate on train and test


In [ ]:
# report loss and accuracy on all the data trained with for dropout regularized final evaluation

In [ ]:
# report loss and accuracy on the testing data with for dropout regularized final evaluation

In [ ]:
# save the final model to file for later evaluation

## Final Report: Dropout Regularized Model Performance

Fill in this table to report the final result you saw for your L2 regularized
model.

| Metric         | Result |
|----------------|--------|
| dropout rate   | 0.00   |
| train loss     | 0.0000 |
| test  loss     | 0.0000 |
| train accuracy | 0.0000 |
| test  accuracy | 0.0000 |

Congratulations on completing this assignment.   Hopefully you remembered to 
make commits after each of the 3 tasks and have pushed them to your GitHub repository for grading.  The autograder will
only be giving 70/70 points for the 5 autograded tasks, I will look that you have completed the asked for plots and visualizations, and
work to fill in the final report tables for the last 4 tasks.

<font color='blue'>
    
**What to remember from this assignment:**

- Keras callbacks are useful to customize your training workflow and monitor the specific information you need for your project.
- Regularization will help you reduce overfitting.
- Regularization will drive your weights to lower values.
- L2 regularization and Dropout are two very effective regularization techniques.
